<h1><center>Naive Bayes - Titanic survival</center></h1>
<h2><center>Probabilistic Machine Learning</center></h2>

### Dependencies

In [ ]:
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats.stats import pearsonr
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score, precision_score
# Suppress warnings
warnings.filterwarnings("ignore")

### Load data

In [ ]:
data = pd.read_csv('titanic_dataset.csv')

### Overview the data

In [ ]:
data.head()

In [ ]:
data.describe()

One advantage of Bayesian models is that it works well enough with small data, having more would give you more accurate probabilities but it's not data hungry as something like deep learning.

### Pre-process
* feature selection, data cleaning, feature engineering and data imputation

In [ ]:
features = ['Age', 'Embarked', 'Fare', 'Parch', 'Pclass', 'Sex', 'SibSp']
target = 'Survived'

# Categorical values need to be transformed into numeric.
data['Sex'] = data['Sex'].replace(["female", "male"], [0, 1])
data['Embarked'] = data['Embarked'].replace(['S', 'C', 'Q'], [1, 2, 3])
data['Age'] = pd.qcut(data['Age'], 10, labels=False)

In [ ]:
# Drop missing values from the train set.
data.dropna(axis=0, inplace=True)
labels = data[target].values

Our processed train set

In [ ]:
data.head()

### Correlation study
* As we saw Naive Bayes models expect the features to be independent, so let's apply the Pearson correlation coefficient on them to give us a hint about how independent they are from the others.

In [ ]:
plt.figure(figsize=(10,10))
sns.heatmap(data.corr(),  annot=True, fmt='.2f', linewidths=.2)
plt.show()

In [ ]:
continuous_numeric_features = ['Age', 'Fare']
for feature in continuous_numeric_features:
    sns.distplot(data[feature])
    plt.show()

### Split data in train and validation (80% ~ 20%)

In [ ]:
labels=data['Survived']
data=data.loc[:,data.columns!='Survived']

In [ ]:
data.drop(columns=['Name','Ticket','Cabin'],inplace=True)

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(data, labels, test_size=0.2, random_state=1)

In [ ]:
X_train, X_val, Y_train, Y_val = train_test_split(X_train, Y_train, test_size=0.2, random_state=1)

In [ ]:
X_train.shape, X_val.shape,X_test.shape

#### How Naive Bayes algorithm works?
* Convert the data set into a frequency table
* Create Likelihood table by finding the probabilities.
* Now, use Naive Bayesian equation to calculate the posterior probability for each class. The class with the highest posterior probability is the outcome of prediction.


##### As you can see the whole model is built upon the probabilities of events, that would be our features.

In [ ]:
classifier = GaussianNB()

#### Fit the first part
* Fitting data here is really fast.

In [ ]:
classifier.fit(X_train, Y_train)

In [ ]:
def cross_validate(classifier, train, validation):
    X_train = train[0]
    Y_train = train[1]
    X_val = validation[0]
    Y_val = validation[1]
    train_predictions = classifier.predict(X_train)
    train_accuracy = accuracy_score(train_predictions, Y_train)
    train_recall = recall_score(train_predictions, Y_train)
    train_precision = precision_score(train_predictions, Y_train)

    val_predictions = classifier.predict(X_val)
    val_accuracy = accuracy_score(val_predictions, Y_val)
    val_recall = recall_score(val_predictions, Y_val)
    val_precision = precision_score(val_predictions, Y_val)

    print('Model metrics')
    print('Accuracy  Train: %.2f, Validation: %.2f' % (train_accuracy, val_accuracy))
    print('Recall    Train: %.2f, Validation: %.2f' % (train_recall, val_recall))
    print('Precision Train: %.2f, Validation: %.2f' % (train_precision, val_precision))

In [ ]:
cross_validate(classifier, (X_train, Y_train), (X_val, Y_val))

### Apply the model on the test data 

In [ ]:
y_pred = classifier.predict(X_test)

In [ ]:
from sklearn import metrics
print("Accuracy :",metrics.accuracy_score(Y_test,y_pred))

In [ ]:
sns.heatmap(metrics.confusion_matrix(Y_test, y_pred),annot=True)
plt.show()